In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL2dvbnphbG92aWRhbC9Eb2N1bWVudHMvR2l0SHViL1B5dGhvbl9mb3JfU3ludGhldGljX0Jpb2xvZ3kvY2hhcHRlcnM='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/Users/gonzalovidal/opt/anaconda3/envs/psb/lib/python3.11/importlib/_bootstrap.py": 1749130019.08049, "/Users/gonzalovidal/opt/anaconda3/envs/psb/lib/python3.11/importlib/_bootstrap_external.py": 1749130019.092533, "/Users/gonzalovidal/opt/anaconda3/envs/psb/lib/python3.11/zipimport.py": 1749130014.228809, "/Users/gonzalovidal/opt/anaconda3/envs/psb/lib/python3.11/codecs.py": 1749130012.791808, "/Users/gonzalovidal/opt/anaconda3/envs/psb/lib/python3.11/encodings/aliases.py": 1749130016.218093, "/Users/gonzalovidal/opt/anaconda3/envs/psb/lib/python3.11/encodings/__init__.py": 1749130016.206209, "/Users/gonzalovidal/opt/anaconda3/envs/psb/lib/python3.11/encodings/utf_8.py": 1749130017.621478, "/Users/gonzalovidal/opt/anaconda3/envs/psb/lib/python3.11/abc.py": 1749130012.596308, "/Users/gonzalovidal/opt/anaconda3/envs/psb/lib/python3.11/io.py": 1749130013.2025, "/Users/gonzalovidal/opt/anaconda3/envs/psb/lib/python3.11/stat.py": 1749130013.847848, "/Users/gonzalovidal/opt/anaconda3/envs

In [2]:
sequence = "ATGCGTACCGTTAG"
sequence

'ATGCGTACCGTTAG'

In [3]:
sequence = "ATGCGTACCGTTAG"

first_base = sequence[0]
start_codon = sequence[0:3]
middle_region = sequence[3:9]
last_three = sequence[-3:]

first_base, start_codon, middle_region, last_three

('A', 'ATG', 'CGTACC', 'TAG')

In [4]:
sequence.startswith("ATG"), sequence.endswith("TAG")

(True, True)

In [5]:
len(sequence)

14

In [6]:
def is_coding_sequence_length(seq: str) -> bool:
    return len(seq) % 3 == 0


is_coding_sequence_length("ATGAAATGA"), is_coding_sequence_length("ATGAAATG")

(True, False)

In [7]:
from collections import Counter

sequence = "ATGCGTACCGTTAG"
base_counts = Counter(sequence)
base_counts

Counter({'T': 4, 'G': 4, 'A': 3, 'C': 3})

In [8]:
def gc_content(seq: str) -> float:
    seq = seq.upper()
    if len(seq) == 0:
        return 0.0
    gc = seq.count("G") + seq.count("C")
    return gc / len(seq)


gc_content(sequence)

0.5

In [9]:
print(f"GC content: {gc_content(sequence):.1%}")

GC content: 50.0%


In [10]:
raw_sequence = " atgcgtaccgttag\n"
clean_sequence = raw_sequence.strip().upper()
clean_sequence

'ATGCGTACCGTTAG'

In [11]:
def clean_dna(seq: str) -> str:
    return "".join(seq.split()).upper()


def is_valid_dna(seq: str) -> bool:
    allowed = set("ATGC")
    cleaned = clean_dna(seq)
    return all(base in allowed for base in cleaned)


is_valid_dna("ATG CCG\nTTA"), is_valid_dna("ATGBCC")

(True, False)

In [12]:
def require_valid_dna(seq: str) -> str:
    cleaned = clean_dna(seq)
    invalid_bases = sorted(set(cleaned) - set("ATGC"))
    if invalid_bases:
        raise ValueError(f"Invalid DNA symbols: {invalid_bases}")
    return cleaned


require_valid_dna("atg ccg tta")

'ATGCCGTTA'

In [13]:
complement_map = str.maketrans("ATGC", "TACG")


def complement(seq: str) -> str:
    seq = require_valid_dna(seq)
    return seq.translate(complement_map)


def reverse_complement(seq: str) -> str:
    seq = require_valid_dna(seq)
    return seq.translate(complement_map)[::-1]


sequence = "ATGCCGTA"
complement(sequence), reverse_complement(sequence)

('TACGGCAT', 'TACGGCAT')

In [14]:
sequence = "ATGGAATTCGCCGTTAA"
sequence.find("GAATTC")

3

In [15]:
sequence.find("GGATCC")

-1

In [16]:
def contains_site(seq: str, site: str) -> bool:
    seq = require_valid_dna(seq)
    site = require_valid_dna(site)
    return site in seq


contains_site(sequence, "GAATTC"), contains_site(sequence, "GGATCC")

(True, False)

In [17]:
restriction_sites = {
    "EcoRI": "GAATTC",
    "BamHI": "GGATCC",
    "BsaI": "GGTCTC",
    "NotI": "GCGGCCGC",
}


def find_present_sites(seq: str, site_map: dict[str, str]) -> list[str]:
    seq = require_valid_dna(seq)
    present = []
    for enzyme, site in site_map.items():
        if site in seq:
            present.append(enzyme)
    return present


find_present_sites("ATGGAATTCGGTCTCTAA", restriction_sites)

['EcoRI', 'BsaI']

In [18]:
def transcribe_dna(seq: str) -> str:
    seq = require_valid_dna(seq)
    return seq.replace("T", "U")


coding_dna = "ATGGCCATTGTAATGGGCCGCTGA"
transcribe_dna(coding_dna)

'AUGGCCAUUGUAAUGGGCCGCUGA'

In [19]:
CODON_TABLE = {
    "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
    "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
    "TAT": "Y", "TAC": "Y", "TAA": "*", "TAG": "*",
    "TGT": "C", "TGC": "C", "TGA": "*", "TGG": "W",
    "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
    "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
    "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
    "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}


def translate_dna(seq: str, stop_at_stop: bool = False) -> str:
    seq = require_valid_dna(seq)
    if len(seq) % 3 != 0:
        raise ValueError("Sequence length must be divisible by 3 for translation.")

    protein = []
    for i in range(0, len(seq), 3):
        codon = seq[i:i + 3]
        amino_acid = CODON_TABLE[codon]
        if stop_at_stop and amino_acid == "*":
            break
        protein.append(amino_acid)
    return "".join(protein)


translate_dna(coding_dna)

'MAIVMGR*'

In [20]:
translate_dna(coding_dna, stop_at_stop=True)

'MAIVMGR'

In [21]:
def looks_like_simple_orf(seq: str) -> bool:
    seq = require_valid_dna(seq)
    if len(seq) < 6:
        return False
    if len(seq) % 3 != 0:
        return False
    if not seq.startswith("ATG"):
        return False
    if seq[-3:] not in {"TAA", "TAG", "TGA"}:
        return False
    internal_codons = [seq[i:i+3] for i in range(3, len(seq) - 3, 3)]
    return all(codon not in {"TAA", "TAG", "TGA"} for codon in internal_codons)


looks_like_simple_orf("ATGGCCATTGTAATGGGCCGCTGA"), looks_like_simple_orf("ATGTAGGCCTAA")

(True, False)

In [22]:
def gc_content_windows(seq: str, window_size: int) -> list[tuple[int, str, float]]:
    seq = require_valid_dna(seq)
    windows = []
    for start in range(0, len(seq) - window_size + 1):
        window = seq[start:start + window_size]
        windows.append((start, window, gc_content(window)))
    return windows


window_results = gc_content_windows("ATGCGCGCTTAA", window_size=4)
window_results[:5]

[(0, 'ATGC', 0.5),
 (1, 'TGCG', 0.75),
 (2, 'GCGC', 1.0),
 (3, 'CGCG', 1.0),
 (4, 'GCGC', 1.0)]

In [23]:
max(window_results, key=lambda row: row[2])

(2, 'GCGC', 1.0)

In [24]:
from pathlib import Path
import tempfile

chapter4_demo_dir = Path(tempfile.mkdtemp(prefix="synbio_ch4_"))
fasta_path = chapter4_demo_dir / "candidate_parts.fasta"

fasta_text = ">promoter_variant_1\nATGCGTACCGTTAG\n>promoter_variant_2\nATGGAATTCGGTCTCTAA\n>coding_variant_1\nATGGCCATTGTAATGGGCCGCTGA\n"

fasta_path.write_text(fasta_text)
str(fasta_path)

'/var/folders/65/dt4l3nw13q57n8x9nlly46640000gn/T/synbio_ch4_zluvn8_f/candidate_parts.fasta'

In [25]:
def read_fasta(path: Path) -> list[dict[str, str]]:
    records = []
    header = None
    sequence_lines = []

    with path.open() as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append({
                        "id": header,
                        "sequence": require_valid_dna("".join(sequence_lines)),
                    })
                header = line[1:]
                sequence_lines = []
            else:
                sequence_lines.append(line)

        if header is not None:
            records.append({
                "id": header,
                "sequence": require_valid_dna("".join(sequence_lines)),
            })

    return records


records = read_fasta(fasta_path)
records

[{'id': 'promoter_variant_1', 'sequence': 'ATGCGTACCGTTAG'},
 {'id': 'promoter_variant_2', 'sequence': 'ATGGAATTCGGTCTCTAA'},
 {'id': 'coding_variant_1', 'sequence': 'ATGGCCATTGTAATGGGCCGCTGA'}]

In [26]:
def summarize_sequence_record(record: dict[str, str]) -> dict[str, object]:
    seq = record["sequence"]
    return {
        "id": record["id"],
        "length": len(seq),
        "gc_percent": round(gc_content(seq) * 100, 1),
        "starts_with_atg": seq.startswith("ATG"),
        "contains_ecori": "GAATTC" in seq,
        "contains_bsai": "GGTCTC" in seq,
        "simple_orf": looks_like_simple_orf(seq),
    }


summaries = [summarize_sequence_record(record) for record in records]
summaries

[{'id': 'promoter_variant_1',
  'length': 14,
  'gc_percent': 50.0,
  'starts_with_atg': True,
  'contains_ecori': False,
  'contains_bsai': False,
  'simple_orf': False},
 {'id': 'promoter_variant_2',
  'length': 18,
  'gc_percent': 38.9,
  'starts_with_atg': True,
  'contains_ecori': True,
  'contains_bsai': True,
  'simple_orf': True},
 {'id': 'coding_variant_1',
  'length': 24,
  'gc_percent': 54.2,
  'starts_with_atg': True,
  'contains_ecori': False,
  'contains_bsai': False,
  'simple_orf': True}]

In [27]:
def passes_simple_design_screen(seq: str) -> bool:
    seq = require_valid_dna(seq)
    gc = gc_content(seq)
    return (
        looks_like_simple_orf(seq)
        and "GAATTC" not in seq
        and "GGTCTC" not in seq
        and 0.35 <= gc <= 0.65
    )


for record in records:
    seq = record["sequence"]
    print(record["id"], passes_simple_design_screen(seq))

promoter_variant_1 False
promoter_variant_2 False
coding_variant_1 True


In [28]:
def design_score(seq: str) -> tuple[bool, float]:
    seq = require_valid_dna(seq)
    passes = passes_simple_design_screen(seq)
    gc_distance_from_target = abs(gc_content(seq) - 0.50)
    return (passes, -gc_distance_from_target)


ranked_records = sorted(
    records,
    key=lambda record: design_score(record["sequence"]),
    reverse=True,
)

[(record["id"], design_score(record["sequence"])) for record in ranked_records]

[('coding_variant_1', (True, -0.04166666666666663)),
 ('promoter_variant_1', (False, -0.0)),
 ('promoter_variant_2', (False, -0.1111111111111111))]

In [29]:
def kmers(seq: str, k: int) -> list[str]:
    seq = require_valid_dna(seq)
    return [seq[i:i+k] for i in range(len(seq) - k + 1)]


kmers("ATGCGTA", 3)

['ATG', 'TGC', 'GCG', 'CGT', 'GTA']

In [30]:
three_mer_counts = Counter(kmers("ATGCGTATGC", 3))
three_mer_counts

Counter({'ATG': 2, 'TGC': 2, 'GCG': 1, 'CGT': 1, 'GTA': 1, 'TAT': 1})

In [31]:
sequence_record = {
    "id": "coding_variant_1",
    "sequence": "ATGGCCATTGTAATGGGCCGCTGA",
    "description": "candidate reporter CDS",
    "source": "design_round_1",
}

sequence_record

{'id': 'coding_variant_1',
 'sequence': 'ATGGCCATTGTAATGGGCCGCTGA',
 'description': 'candidate reporter CDS',
 'source': 'design_round_1'}

In [32]:
import csv

report_rows = []

for record in records:
    seq = record["sequence"]
    report_rows.append({
        "id": record["id"],
        "length": len(seq),
        "gc_percent": round(gc_content(seq) * 100, 1),
        "passes_screen": passes_simple_design_screen(seq),
        "protein": translate_dna(seq, stop_at_stop=True) if looks_like_simple_orf(seq) else "",
    })

report_rows

[{'id': 'promoter_variant_1',
  'length': 14,
  'gc_percent': 50.0,
  'passes_screen': False,
  'protein': ''},
 {'id': 'promoter_variant_2',
  'length': 18,
  'gc_percent': 38.9,
  'passes_screen': False,
  'protein': 'MEFGL'},
 {'id': 'coding_variant_1',
  'length': 24,
  'gc_percent': 54.2,
  'passes_screen': True,
  'protein': 'MAIVMGR'}]

In [33]:
report_path = chapter4_demo_dir / "sequence_report.csv"

with report_path.open("w", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["id", "length", "gc_percent", "passes_screen", "protein"],
    )
    writer.writeheader()
    writer.writerows(report_rows)

report_path.exists(), report_path.name

(True, 'sequence_report.csv')

In [34]:
report_path.read_text()

'id,length,gc_percent,passes_screen,protein\npromoter_variant_1,14,50.0,False,\npromoter_variant_2,18,38.9,False,MEFGL\ncoding_variant_1,24,54.2,True,MAIVMGR\n'

In [35]:
from Bio.Seq import Seq

bio_seq = Seq("ATGGCCATTGTAATGGGCCGCTGA")

len(bio_seq), bio_seq[:6], bio_seq.reverse_complement()

(24, Seq('ATGGCC'), Seq('TCAGCGGCCCATTACAATGGCCAT'))

In [36]:
bio_seq.transcribe(), bio_seq.translate(), bio_seq.translate(to_stop=True)

(Seq('AUGGCCAUUGUAAUGGGCCGCUGA'), Seq('MAIVMGR*'), Seq('MAIVMGR'))

In [37]:
from Bio.SeqRecord import SeqRecord

record = SeqRecord(
    bio_seq,
    id="demo_cds_1",
    name="demo_cds_1",
    description="Synthetic demonstration coding sequence",
)
record.annotations["source"] = "chapter4_demo"

record.id, record.description, str(record.seq)

('demo_cds_1',
 'Synthetic demonstration coding sequence',
 'ATGGCCATTGTAATGGGCCGCTGA')

In [38]:
from Bio import SeqIO

bio_records = list(SeqIO.parse(fasta_path, "fasta"))
[(record.id, len(record), str(record.seq[:6])) for record in bio_records]

[('promoter_variant_1', 14, 'ATGCGT'),
 ('promoter_variant_2', 18, 'ATGGAA'),
 ('coding_variant_1', 24, 'ATGGCC')]

In [39]:
first_bio_record = bio_records[0]
first_bio_record.id, first_bio_record.description, type(first_bio_record.seq).__name__

('promoter_variant_1', 'promoter_variant_1', 'Seq')

In [40]:
record_dict = SeqIO.to_dict(bio_records)
record_dict["coding_variant_1"].seq.translate(to_stop=True)

Seq('MAIVMGR')

In [41]:
biopython_filtered_path = chapter4_demo_dir / "no_ecori_records.fasta"
filtered_records = [record for record in bio_records if "GAATTC" not in record.seq]
SeqIO.write(filtered_records, biopython_filtered_path, "fasta")

biopython_filtered_path.name, biopython_filtered_path.exists()

('no_ecori_records.fasta', True)

In [42]:
biopython_filtered_path.read_text()

'>promoter_variant_1\nATGCGTACCGTTAG\n>coding_variant_1\nATGGCCATTGTAATGGGCCGCTGA\n'

In [43]:
genbank_path = chapter4_demo_dir / "demo_construct.gb"

genbank_text = """LOCUS       DEMO0001                  39 bp    DNA     linear   SYN 01-JAN-2024
DEFINITION  Synthetic demonstration construct.
ACCESSION   DEMO0001
VERSION     DEMO0001.1
KEYWORDS    synthetic biology.
SOURCE      synthetic DNA construct
  ORGANISM  synthetic DNA construct
            other sequences; artificial sequences.
FEATURES             Location/Qualifiers
     source          1..39
                     /organism=\"synthetic DNA construct\"
                     /mol_type=\"other DNA\"
     promoter        1..9
                     /label=\"P_demo\"
     CDS             10..33
                     /gene=\"demoGFP\"
                     /product=\"demo protein\"
                     /codon_start=1
                     /translation=\"MAIVMGR\"
ORIGIN
        1 ttgacatat atggccatt gtaatgggcc gctgaaatata
//
"""

genbank_path.write_text(genbank_text)
genbank_record = SeqIO.read(genbank_path, "genbank")

genbank_record.id, len(genbank_record), genbank_record.description

('DEMO0001.1', 39, 'Synthetic demonstration construct')

In [44]:
sorted(genbank_record.annotations.keys())

['accessions',
 'data_file_division',
 'date',
 'keywords',
 'molecule_type',
 'organism',
 'sequence_version',
 'source',
 'taxonomy',
 'topology']

In [45]:
[(feature.type, int(feature.location.start), int(feature.location.end)) for feature in genbank_record.features]

[('source', 0, 39), ('promoter', 0, 9), ('CDS', 9, 33)]

In [46]:
cds_feature = next(feature for feature in genbank_record.features if feature.type == "CDS")
str(cds_feature.extract(genbank_record.seq)), cds_feature.qualifiers["gene"][0], cds_feature.qualifiers["translation"][0]

('ATGGCCATTGTAATGGGCCGCTGA', 'demoGFP', 'MAIVMGR')

In [47]:
from Bio.SeqFeature import SeqFeature, FeatureLocation

custom_record = SeqRecord(
    Seq("ATGGCCATTGTAATGGGCCGCTGA"),
    id="design_001",
    name="design_001",
    description="Custom design record generated in Python",
)
custom_record.annotations["molecule_type"] = "DNA"
custom_record.features = [
    SeqFeature(FeatureLocation(0, len(custom_record.seq)), type="CDS", qualifiers={"gene": ["demo_gene"]})
]

custom_record.id, custom_record.features[0].type, str(custom_record.seq.translate())

('design_001', 'CDS', 'MAIVMGR*')

In [48]:
from Bio.Blast import NCBIWWW

NCBIWWW.email = "your_email@example.com"

with NCBIWWW.qblast("blastn", "nt", str(record_dict["coding_variant_1"].seq)) as result_handle:
    blast_xml = result_handle.read()

# Save the XML for later parsing
(chapter4_demo_dir / "coding_variant_1_blast.xml").write_text(blast_xml)

1995

In [49]:
from Bio import SearchIO

blast_tab_path = chapter4_demo_dir / "toy_blast.tsv"
blast_tab_text = """query1	subjectA	100.000	12	0	0	1	12	5	16	1e-20	50.0
query1	subjectB	91.667	12	1	0	1	12	20	31	2e-10	40.0
query2	subjectA	87.500	8	1	0	1	8	30	37	1e-05	25.0
"""
blast_tab_path.write_text(blast_tab_text)

qresults = list(SearchIO.parse(blast_tab_path, "blast-tab"))
[(qresult.id, len(qresult)) for qresult in qresults]

[('query1', 2), ('query2', 1)]

In [50]:
first_qresult = qresults[0]
best_hit = first_qresult[0]
best_hsp = best_hit.hsps[0]

best_hit.id, best_hsp.evalue, best_hsp.ident_pct

('subjectA', 1e-20, 100.0)

In [51]:
best_hsp.query_start, best_hsp.query_end, best_hsp.hit_start, best_hsp.hit_end

(0, 12, 4, 16)

In [52]:
def summarize_blast_qresults(qresults):
    rows = []
    for qresult in qresults:
        for hit in qresult:
            top_hsp = hit.hsps[0]
            rows.append(
                {
                    "query_id": qresult.id,
                    "hit_id": hit.id,
                    "evalue": top_hsp.evalue,
                    "bitscore": top_hsp.bitscore,
                    "identity_percent": round(top_hsp.ident_pct, 1),
                    "query_start": top_hsp.query_start,
                    "query_end": top_hsp.query_end,
                    "hit_start": top_hsp.hit_start,
                    "hit_end": top_hsp.hit_end,
                }
            )
    return rows

blast_summary_rows = summarize_blast_qresults(qresults)
blast_summary_rows

[{'query_id': 'query1',
  'hit_id': 'subjectA',
  'evalue': 1e-20,
  'bitscore': 50.0,
  'identity_percent': 100.0,
  'query_start': 0,
  'query_end': 12,
  'hit_start': 4,
  'hit_end': 16},
 {'query_id': 'query1',
  'hit_id': 'subjectB',
  'evalue': 2e-10,
  'bitscore': 40.0,
  'identity_percent': 91.7,
  'query_start': 0,
  'query_end': 12,
  'hit_start': 19,
  'hit_end': 31},
 {'query_id': 'query2',
  'hit_id': 'subjectA',
  'evalue': 1e-05,
  'bitscore': 25.0,
  'identity_percent': 87.5,
  'query_start': 0,
  'query_end': 8,
  'hit_start': 29,
  'hit_end': 37}]